In [1]:
!pip install skorch -q

import glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 4.8 MB/s eta 0:00:00
cuda


In [2]:
#vérification du chemin des fichiers
import os
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/shivamb/machine-predictive-maintenance-classification/predictive_maintenance.csv


In [3]:
path = glob.glob("/kaggle/input/datasets/shivamb/machine-predictive-maintenance-classification/predictive_maintenance.csv", recursive=True)[0]
print(path)
df = pd.read_csv(path)
print(df.shape)
df.info()
df.head()

/kaggle/input/datasets/shivamb/machine-predictive-maintenance-classification/predictive_maintenance.csv
(10000, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Target                   10000 non-null  int64  
 9   Failure Type             10000 non-null  object 
dtypes: float64(3), int64(4), object(3)
memory usage: 781.4+ KB


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [4]:
print(df.isnull().sum())
print("Lignes dupliquées :", df.duplicated().sum())
print("Product ID dupliqués :", df["Product ID"].duplicated().sum())
print(df["Failure Type"].value_counts())
print(pd.crosstab(df["Target"], df["Failure Type"]))

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Target                     0
Failure Type               0
dtype: int64
Lignes dupliquées : 0
Product ID dupliqués : 0
Failure Type
No Failure                  9652
Heat Dissipation Failure     112
Power Failure                 95
Overstrain Failure            78
Tool Wear Failure             45
Random Failures               18
Name: count, dtype: int64
Failure Type  Heat Dissipation Failure  No Failure  Overstrain Failure  \
Target                                                                   
0                                    0        9643                   0   
1                                  112           9                  78   

Failure Type  Power Failure  Random Failures  Tool Wear Failure  
Target                                            

In [5]:
# 1. Renommons les colonnes (plus simples à manipuler)
df = df.rename(columns={
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rot_speed",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
})

# 2. Supprimer les lignes aux étiquettes contradictoires (panne signalée mais type "No Failure")
mask = (df["Target"] == 1) & (df["Failure Type"] == "No Failure")
print("Lignes incohérentes supprimées :", mask.sum())
df = df[~mask].reset_index(drop=True)

# 3. Supprimer les colonnes inutiles ou dangereuses
#    UDI, Product ID : simples identifiants
#    Target : dérivée de Failure Type, la garder serait une fuite de données (data leakage)
df = df.drop(columns=["UDI", "Product ID", "Target"])

# 4. Encoder Type (qualité du produit : L < M < H)
df["Type"] = df["Type"].map({"L": 0, "M": 1, "H": 2})

# 5. Encoder la cible multi-classes
le = LabelEncoder()
df["label"] = le.fit_transform(df["Failure Type"])
print(dict(zip(le.classes_, le.transform(le.classes_))))
df.head()

Lignes incohérentes supprimées : 9
{'Heat Dissipation Failure': np.int64(0), 'No Failure': np.int64(1), 'Overstrain Failure': np.int64(2), 'Power Failure': np.int64(3), 'Random Failures': np.int64(4), 'Tool Wear Failure': np.int64(5)}


,Type,air_temp,process_temp,rot_speed,torque,tool_wear,Failure Type,label
0,1,298.1,308.6,1551,42.8,0,No Failure,1
1,0,298.2,308.7,1408,46.3,3,No Failure,1
2,0,298.1,308.5,1498,49.4,5,No Failure,1
3,0,298.2,308.6,1433,39.5,7,No Failure,1
4,0,298.2,308.7,1408,40.0,9,No Failure,1


In [6]:
features = ["Type", "air_temp", "process_temp", "rot_speed", "torque", "tool_wear"]
X, y = df[features], df["label"]

# Split STRATIFIÉ : chaque classe garde sa proportion dans train et test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Standardisation : on apprend (fit) sur le train uniquement
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(X_train_s.shape, X_test_s.shape)
print(y_train.value_counts().sort_index())

(7992, 6) (1999, 6)
label
0      90
1    7714
2      62
3      76
4      14
5      36
Name: count, dtype: int64
